=============================================================================
**PHASE 3 — NOTEBOOK 1: LIGHTNING AI SETUP + MODEL B TRAINING**  
=============================================================================
**RUN THIS ON: Lightning AI Studio (T4 GPU)**  
NOT on your MacBook — this requires CUDA.

**WHAT THIS NOTEBOOK DOES:**  
  1. Verify GPU is available and check VRAM
  2. Install all dependencies
  3. Upload/mount HAM10000 dataset
  4. Train EfficientNet-B3 (Model B upgrade from Phase 2's B1)
  5. Save best checkpoint

**ESTIMATED TIME ON T4: ~45-60 minutes total**  
  Stage 1 (frozen, 5 epochs):  ~8 minutes
  Stage 2 (unfrozen, 25 epochs): ~40 minutes

**SAVE BUDGET: uses ~3-4 of your 22 GPU hours**  
=============================================================================

### CELL 1: GPU VERIFICATION

In [1]:
# Run this FIRST. If CUDA is not available, stop and check
# your Lightning AI studio configuration.

In [2]:
import torch
import os, sys

print("=" * 55)
print("PHASE 3 — LIGHTNING AI SETUP")
print("=" * 55)

# Must be True for Phase 3 to work
print(f"\nCUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM:            {vram:.1f} GB")
    print(f"\n✅ GPU ready — Phase 3 can proceed")
    device = torch.device('cuda')
else:
    print("\n❌ CUDA not available")
    print("   Go to Lightning AI Studio settings")
    print("   Select machine type: T4 GPU")
    print("   Then restart this notebook")
    device = torch.device('cpu')

print(f"\nDevice: {device}")

PHASE 3 — LIGHTNING AI SETUP

CUDA available:  True
GPU:             Tesla T4
VRAM:            15.6 GB

✅ GPU ready — Phase 3 can proceed

Device: cuda


In [3]:
# CELL 2: INSTALL DEPENDENCIES
# Run once per session — Lightning AI resets packages on restart

In [4]:
import subprocess

packages = [
    "torch torchvision --index-url https://download.pytorch.org/whl/cu118",
    "timm efficientnet-pytorch",
    "scikit-learn scikit-image",
    "matplotlib seaborn pandas numpy",
    "opencv-python-headless",
    "reportlab nbformat",
]

print("Installing dependencies...")
for pkg in packages:
    result = subprocess.run(
        f"pip install {pkg} -q",
        shell=True, capture_output=True, text=True
    )
    name = pkg.split()[0]
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {name}")

print("\nAll packages installed.")

Installing dependencies...
  ✅ torch
  ✅ timm
  ✅ scikit-learn
  ✅ matplotlib
  ✅ opencv-python-headless
  ✅ reportlab

All packages installed.


### CELL 3: DATASET SETUP

In [5]:
# Option A: Upload via Kaggle API (recommended, fastest)
# Option B: Direct upload through Lightning AI file panel

### ----------------------------------------------------------

In [6]:
# OPTION A: Kaggle API
# 1. Go to kaggle.com → Account → Create API Token
# 2. Download kaggle.json
# 3. Upload kaggle.json to Lightning AI studio
# Then run this cell:
# ----------------------------------------------------------

import os

KAGGLE_JSON_PATH = "/teamspace/studios/this_studio/kaggle.json"   # adjust if different

if os.path.exists(KAGGLE_JSON_PATH):
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    os.system(f"cp {KAGGLE_JSON_PATH} ~/.kaggle/kaggle.json")
    os.system("chmod 600 ~/.kaggle/kaggle.json")
    print("Kaggle credentials set ✅")

    print("\nDownloading HAM10000...")
    os.system(
        "kaggle datasets download -d "
        "kmader/skin-lesion-analysis-toward-melanoma-detection "
        "--path /teamspace/studios/this_studio/dataset_zip -q"
    )
    print("Download complete")

    print("\nExtracting images...")
    os.makedirs("/teamspace/studios/this_studio/dataset/images", exist_ok=True)
    os.system(
        "unzip -q /teamspace/studios/this_studio/dataset_zip/*.zip "
        "-d /teamspace/studios/this_studio/dataset_raw/ 2>/dev/null || true"
    )
    # Copy all jpg files to images folder
    os.system("find /teamspace/studios/this_studio/dataset_raw -name '*.jpg' "
              "-exec cp {} /teamspace/studios/this_studio/dataset/images/ \\;")
    os.system("find /teamspace/studios/this_studio/dataset_raw -name '*.csv' "
              "-exec cp {} /teamspace/studios/this_studio/dataset/ \\;")

    n_imgs = len(os.listdir("/teamspace/studios/this_studio/dataset/images"))
    print(f"Images in dataset folder: {n_imgs:,}")

else:
    print("kaggle.json not found.")
    print("\nOption B — Manual upload:")
    print("  1. Upload HAM10000_metadata.csv to /teamspace/studios/this_studio/dataset/")
    print("  2. Upload all .jpg images to /teamspace/studios/this_studio/dataset/images/")
    print("  3. Re-run this cell to verify")

# Verify paths
METADATA_CSV = "/teamspace/studios/this_studio/dataset/HAM10000_metadata.csv"
IMAGES_DIR   = "/teamspace/studios/this_studio/dataset/images"
OUTPUTS_DIR  = "/teamspace/studios/this_studio/outputs"

os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(f"{OUTPUTS_DIR}/checkpoints", exist_ok=True)

csv_ok  = os.path.exists(METADATA_CSV)
imgs_ok = os.path.isdir(IMAGES_DIR) and len(os.listdir(IMAGES_DIR)) > 5000

print(f"\nMetadata CSV:  {'✅' if csv_ok  else '❌'} {METADATA_CSV}")
print(f"Images folder: {'✅' if imgs_ok else '❌'} {IMAGES_DIR}")

kaggle.json not found.

Option B — Manual upload:
  1. Upload HAM10000_metadata.csv to /teamspace/studios/this_studio/dataset/
  2. Upload all .jpg images to /teamspace/studios/this_studio/dataset/images/
  3. Re-run this cell to verify

Metadata CSV:  ✅ /teamspace/studios/this_studio/dataset/HAM10000_metadata.csv
Images folder: ✅ /teamspace/studios/this_studio/dataset/images


### CELL 4: ADD PROJECT CODE TO PATH

In [7]:
# Upload your Phase 2 code folder to Lightning AI.
# The data/, models/, utils/ folders must be accessible.

### If you uploaded your project as a zip, unzip it first:

In [8]:
# os.system("unzip robust_medical_vision.zip -d /teamspace/studios/this_studio/")

PROJECT_ROOT = "/teamspace/studios/this_studio/robust_medical_vision"
sys.path.insert(0, PROJECT_ROOT)

# Verify imports work
try:
    from data.dataset    import load_metadata, split_dataset, get_dataloaders
    from models.architecture_v2 import (RobustMedicalClassifierV2,
                                         model_summary_v2)
    from models.losses   import CombinedLoss
    print("✅ All project imports successful")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("   Make sure your project code is at /teamspace/studios/this_studio/robust_medical_vision/")

✅ All project imports successful


### CELL 5: DATA PIPELINE

In [9]:
# Same GroupShuffleSplit from Phase 2 — no changes here.
# The only difference: batch_size = 64 instead of 16.
# WHY 64: T4 has 16GB VRAM, EfficientNet-B3 needs ~2.5GB per step.
# 16GB / 2.5GB = ~6 steps in parallel → batch 64 is safe and stable.
# Larger batches = more stable gradients = better training.

In [10]:
import warnings
warnings.filterwarnings('ignore')
from collections import Counter

print("\nLoading HAM10000 metadata...")
df = load_metadata(METADATA_CSV, IMAGES_DIR)
print(f"Loaded {len(df):,} images")

print("\nSplitting by lesion_id (GroupShuffleSplit)...")
df_train, df_val, df_test = split_dataset(df)
print(f"  Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")

# BATCH SIZE 64 — the key Phase 3 upgrade
BATCH_SIZE = 64
print(f"\nBuilding DataLoaders (batch_size={BATCH_SIZE})...")
train_loader, val_loader, test_loader = get_dataloaders(
    df_train, df_val, df_test, batch_size=BATCH_SIZE
)
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")

class_counts = dict(Counter(df_train['label'].values))


Loading HAM10000 metadata...
STEP 1B: Loading HAM10000 Metadata
  Total records in CSV: 10015
  Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']
  Unique lesions (lesion_id): 7470
  Unique images (image_id):   10015
  All 10015 images found on disk. ✅

  Class distribution:
    akiec  (Actinic Keratosis        ):   327 (3.3%)
    bcc    (Basal Cell Carcinoma     ):   514 (5.1%)
    bkl    (Benign Keratosis         ):  1099 (11.0%)
    df     (Dermatofibroma           ):   115 (1.1%)
    mel    (Melanoma                 ):  1113 (11.1%)
    nv     (Melanocytic Nevus        ):  6705 (66.9%)
    vasc   (Vascular Lesion          ):   142 (1.4%)
Loaded 10,015 images

Splitting by lesion_id (GroupShuffleSplit)...

STEP 1C: Group-Based Train/Val/Test Split
WHY: Splitting by lesion_id prevents same lesion in train+test
  Train:  6959 images | 5228 unique lesions
  Val:    1529 images | 1121 unique lesions
  Test:   1527 images | 1121 unique lesions


In [11]:
# CELL 6: BUILD MODEL B (EfficientNet-B3)
# First time this runs, downloads B3 ImageNet weights (~47MB).

In [12]:
print("\nBuilding Model B — EfficientNet-B3...")

model = RobustMedicalClassifierV2(
    num_classes=7,
    freeze_backbone=True   # Stage 1: frozen backbone
)
model = model.to(device)
model_summary_v2(model)

# Quick sanity check: one forward pass
test_batch = torch.randn(4, 3, 224, 224).to(device)
output     = model(test_batch)
print(f"\nForward pass check:")
print(f"  logits shape:   {output['logits'].shape}   ✅")
print(f"  alpha shape:    {output['alpha'].shape}    ✅")
print(f"  features shape: {output['features'].shape} ✅")


Building Model B — EfficientNet-B3...
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /teamspace/studios/this_studio/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 225MB/s]


  Backbone: EfficientNet-B3 — FROZEN (Stage 1)

MODEL SUMMARY — Phase 3 (EfficientNet-B3)
  Total parameters:       11,619,126
  Trainable:                 922,894
  Frozen:                 10,696,232

  Backbone:  EfficientNet-B3
  Features:  1536-dim → head → 256-dim
  Head:      Linear(1536→512) + BN + GELU + MCDrop(0.4)
             Linear(512→256)  + GELU + MCDrop(0.3)
  Output 1:  Standard classifier → 7 logits
  Output 2:  Evidential head    → 7 Dirichlet α
  MC passes: 30 at inference (T4 GPU, ~0.3s/image)

Forward pass check:
  logits shape:   torch.Size([4, 7])   ✅
  alpha shape:    torch.Size([4, 7])    ✅
  features shape: torch.Size([4, 256]) ✅


In [13]:
# CELL 7: LOSS FUNCTION
# Identical to Phase 2 — Focal + Evidential combined loss.

In [14]:
from models.losses import CombinedLoss

loss_fn = CombinedLoss(
    num_classes  = 7,
    gamma        = 2.0,
    lambda_ev    = 0.5,
    class_counts = class_counts,
    device       = device,
)
print("Loss function ready ✅")

  Focal alpha (per-class weights): [0.007 0.041 0.041 0.09  0.142 0.338 0.342]
  Combined Loss: Focal(γ=2.0) + 0.5×Evidential
Loss function ready ✅


### CELL 8: STAGE 1 TRAINING — FROZEN BACKBONE

In [15]:
# Only the head trains. Backbone weights stay frozen.
# WHY: head needs to adapt to skin lesions before we touch backbone.
# Expected: 8 minutes on T4. F1 goes from random (~0.14) to ~0.5

In [16]:
from models.trainer import (get_device, train, validate,
                              compute_metrics, build_optimizer,
                              build_scheduler)
from models.trainer import EarlyStopping
import time

STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 25

print("\n" + "=" * 55)
print("STAGE 1: Frozen Backbone Training")
print(f"  Epochs: {STAGE1_EPOCHS}")
print(f"  Only head layers train — backbone frozen")
print("=" * 55)

# Higher LR for head in Stage 1 — learning from scratch
optimizer_s1 = build_optimizer(
    model, head_lr=1e-3, backbone_lr=0.0
)
scheduler_s1 = build_scheduler(
    optimizer_s1, T_0=STAGE1_EPOCHS
)
early_stop_s1 = EarlyStopping(
    patience=4,
    checkpoint_path=f"{OUTPUTS_DIR}/checkpoints/stage1_best.pth"
)

s1_start = time.time()
for epoch in range(1, STAGE1_EPOCHS + 1):
    print(f"\n  Epoch {epoch}/{STAGE1_EPOCHS}")
    t0 = time.time()

    from models.trainer import train_one_epoch
    train_m = train_one_epoch(
        model, train_loader, optimizer_s1, scheduler_s1,
        loss_fn, device, epoch, STAGE1_EPOCHS, accumulation_steps=1
        # accumulation_steps=1 because batch 64 is already large enough
    )
    val_m = validate(model, val_loader, loss_fn, device, epoch, STAGE1_EPOCHS)

    print(f"  Train F1: {train_m['f1_macro']:.4f} | "
          f"Val F1: {val_m['f1_macro']:.4f} | "
          f"Val AUROC: {val_m['auroc']:.4f} | "
          f"Time: {time.time()-t0:.0f}s")

    if early_stop_s1.step(val_m['f1_macro'], model, optimizer_s1, epoch):
        break

s1_time = time.time() - s1_start
print(f"\nStage 1 complete in {s1_time/60:.1f} minutes")


STAGE 1: Frozen Backbone Training
  Epochs: 5
  Only head layers train — backbone frozen

  Optimizer: AdamW
    Backbone LR: 0.0 (frozen in Stage 1, gentle in Stage 2)
    Head LR:     0.001
    Weight decay: 0.0001
  Scheduler: CosineAnnealingWarmRestarts(T_0=5, T_mult=1)

  Epoch 1/5
    Batch 50/108 | Loss: 0.9940 | Focal: 0.0861 | Ev: 1.8157
    Batch 100/108 | Loss: 0.9440 | Focal: 0.0863 | Ev: 1.7154
  Train F1: 0.2705 | Val F1: 0.1495 | Val AUROC: 0.7639 | Time: 69s
    ✅ New best val F1: 0.1495 — checkpoint saved

  Epoch 2/5
    Batch 50/108 | Loss: 0.8990 | Focal: 0.0527 | Ev: 1.6926
    Batch 100/108 | Loss: 0.9102 | Focal: 0.0758 | Ev: 1.6686
  Train F1: 0.3927 | Val F1: 0.2274 | Val AUROC: 0.8235 | Time: 64s
    ✅ New best val F1: 0.2274 — checkpoint saved

  Epoch 3/5
    Batch 50/108 | Loss: 0.9258 | Focal: 0.0679 | Ev: 1.7158
    Batch 100/108 | Loss: 0.8991 | Focal: 0.0713 | Ev: 1.6557
  Train F1: 0.4118 | Val F1: 0.2452 | Val AUROC: 0.8175 | Time: 64s
    ✅ New best

### CELL 9: STAGE 2 TRAINING — FULL FINE-TUNING

In [17]:
# Load best Stage 1 weights, then unfreeze backbone.
# Backbone gets 10× smaller LR than head to prevent forgetting.
# Expected: ~40 minutes on T4. Final F1 ~0.72-0.76

In [19]:
print("\n" + "=" * 55)
print("STAGE 2: Full Fine-Tuning (Backbone Unfrozen)")
print("  Backbone LR: 5e-5 (10× smaller than head)")
print("  Reason: gentle update to preserve ImageNet features")
print("=" * 55)

# Load best Stage 1 checkpoint before unfreezing
ckpt = torch.load(
    f"{OUTPUTS_DIR}/checkpoints/stage1_best.pth",
    map_location=device,
    weights_only=False
)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded Stage 1 best (val F1: {ckpt['val_f1']:.4f})")

model.unfreeze_backbone()

# Discriminative learning rates — backbone much smaller than head
optimizer_s2 = build_optimizer(
    model, head_lr=5e-4, backbone_lr=5e-5
)
scheduler_s2 = build_scheduler(
    optimizer_s2, T_0=10
)
early_stop_s2 = EarlyStopping(
    patience=7,
    checkpoint_path=f"{OUTPUTS_DIR}/checkpoints/best_model_b3.pth"
)

s2_start = time.time()
for epoch in range(STAGE1_EPOCHS + 1,
                   STAGE1_EPOCHS + STAGE2_EPOCHS + 1):
    print(f"\n  Epoch {epoch}/{STAGE1_EPOCHS + STAGE2_EPOCHS}")
    t0 = time.time()

    train_m = train_one_epoch(
        model, train_loader, optimizer_s2, scheduler_s2,
        loss_fn, device, epoch,
        STAGE1_EPOCHS + STAGE2_EPOCHS,
        accumulation_steps=1
    )
    val_m = validate(
        model, val_loader, loss_fn, device,
        epoch, STAGE1_EPOCHS + STAGE2_EPOCHS
    )

    print(f"  Train F1: {train_m['f1_macro']:.4f} | "
          f"Val F1: {val_m['f1_macro']:.4f} | "
          f"AUROC: {val_m['auroc']:.4f} | "
          f"Time: {time.time()-t0:.0f}s")

    if early_stop_s2.step(val_m['f1_macro'], model, optimizer_s2, epoch):
        break

s2_time = time.time() - s2_start
print(f"\nStage 2 complete in {s2_time/60:.1f} minutes")
print(f"Total training time: {(s1_time+s2_time)/60:.1f} minutes")
print(f"\nBest model saved: {OUTPUTS_DIR}/checkpoints/best_model_b3.pth")


STAGE 2: Full Fine-Tuning (Backbone Unfrozen)
  Backbone LR: 5e-5 (10× smaller than head)
  Reason: gentle update to preserve ImageNet features
Loaded Stage 1 best (val F1: 0.2582)
  Backbone UNFROZEN — use 10× smaller LR for backbone params

  Optimizer: AdamW
    Backbone LR: 5e-05 (frozen in Stage 1, gentle in Stage 2)
    Head LR:     0.0005
    Weight decay: 0.0001
  Scheduler: CosineAnnealingWarmRestarts(T_0=10, T_mult=1)

  Epoch 6/30
    Batch 50/108 | Loss: 0.7780 | Focal: 0.0592 | Ev: 1.4378
    Batch 100/108 | Loss: 0.8325 | Focal: 0.0574 | Ev: 1.5503
  Train F1: 0.4998 | Val F1: 0.2958 | AUROC: 0.8594 | Time: 75s
    ✅ New best val F1: 0.2958 — checkpoint saved

  Epoch 7/30
    Batch 50/108 | Loss: 0.7679 | Focal: 0.0381 | Ev: 1.4597
    Batch 100/108 | Loss: 0.7428 | Focal: 0.0331 | Ev: 1.4192
  Train F1: 0.5329 | Val F1: 0.3167 | AUROC: 0.8550 | Time: 75s
    ✅ New best val F1: 0.3167 — checkpoint saved

  Epoch 8/30
    Batch 50/108 | Loss: 0.7177 | Focal: 0.0365 | Ev:

In [20]:
# CELL 10: QUICK TEST EVALUATION
# Validate the trained model before moving to Phase 3 pipeline.

In [22]:
import torch.nn.functional as F
import numpy as np

print("\nLoading best checkpoint for evaluation...")
ckpt  = torch.load(
    f"{OUTPUTS_DIR}/checkpoints/best_model_b3.pth",
    map_location=device,
    weights_only=False
)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Epoch {ckpt['epoch']} — Val F1: {ckpt['val_f1']:.4f}")

# Standard test set evaluation
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        out    = model(images)
        probs  = F.softmax(out['logits'], dim=1)
        preds  = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

metrics = compute_metrics(all_preds, all_labels, all_probs)

print(f"\n  Model B (EfficientNet-B3) Test Results:")
print(f"    F1 (macro): {metrics['f1_macro']:.4f}  "
      f"(Phase 2 B1 was 0.569)")
print(f"    AUROC:      {metrics['auroc']:.4f}  "
      f"(Phase 2 was 0.939)")

class_codes = ['nv','mel','bkl','bcc','akiec','vasc','df']
print(f"\n  Per-class F1:")
for cls, f1 in zip(class_codes, metrics['f1_per_class']):
    bar = '█' * int(f1 * 25)
    print(f"    {cls:<8}: {f1:.3f}  {bar}")

print("\n→ NEXT: Run notebook 02_temperature_and_ood.ipynb")


Loading best checkpoint for evaluation...
Epoch 28 — Val F1: 0.6150

  Model B (EfficientNet-B3) Test Results:
    F1 (macro): 0.5754  (Phase 2 B1 was 0.569)
    AUROC:      0.9159  (Phase 2 was 0.939)

  Per-class F1:
    nv      : 0.738  ██████████████████
    mel     : 0.463  ███████████
    bkl     : 0.575  ██████████████
    bcc     : 0.647  ████████████████
    akiec   : 0.495  ████████████
    vasc    : 0.710  █████████████████
    df      : 0.400  ██████████

→ NEXT: Run notebook 02_temperature_and_ood.ipynb
